# Video preparation for DeepLabCut
Crop each video to an **area of interest (ROI)**, black out (or colour, or make transparent) everything outside it, reduce the **frame rate** and **resolution**, and save all videos in **one common format**.

### What you need (one-time install)
In a terminal, in the same environment you run Jupyter from:
```
pip install ipympl ipywidgets pillow tqdm imageio-ffmpeg
```
then restart Jupyter / VS Code once. (ffmpeg is used from your system if installed, otherwise from `imageio-ffmpeg`.)

### How to use it — run the cells from top to bottom
| step | what you do | what happens |
|---|---|---|
| **0 · Settings** | edit the values if needed, run the cell | a folder dialog opens if `INPUT_DIR` is empty |
| **Core functions** | just run | lists the videos found |
| **1 · Draw the ROI** | for each video: pick a shape, click on the image, press **Save & next** | ROIs saved to `rois.json` in the input folder (you can stop and come back later) |
| *Check the ROIs* | optional, just run | thumbnails with the ROI drawn on them |
| **2 · Process** | just run, wait | the new videos are written to the output folder |
| **3 · Check** | just run | a frame of every new video, with size / fps / file size |

**Output:** `<input folder>/DLC_ready/` with one video per input (same file name) and `processing_params.json` (crop/scale used for each video).
Re-running is safe: videos that already have an ROI or an output file are skipped (see `REDO` / `OVERWRITE`).

## 0 · Settings

In [ ]:
# ======================= SETTINGS =======================
# --- folders ---
INPUT_DIR  = ""            # folder with the videos; leave "" to choose it in a dialog
OUTPUT_DIR = ""            # where the new videos go; "" -> <INPUT_DIR>/DLC_ready
VIDEO_EXTS = {".mp4", ".mov", ".avi", ".mkv", ".m4v"}   # which files count as videos

# --- area of interest (ROI) ---
ROI_SHAPE  = "circle"      # default shape in Step 1: "circle", "rectangle", "square" or "polygon" (custom)
                           # (you can still change it for each video in Step 1)

# --- output format ---
TARGET_FPS = 10            # frames per second of the new videos (extra frames are dropped)
OUTPUT_W, OUTPUT_H = 720, 720   # size of the new videos in pixels (even numbers). The ROI is scaled to fit
                                # and centred. Use a square (720x720) for circles/squares,
                                # e.g. 1280x720 for wide rectangles.
UNIFORM_SIZE = True        # True : every video is exactly OUTPUT_W x OUTPUT_H, ROI centred (needed for DLC)
                           # False: just the crop – each video is only as big as its ROI (fitted inside
                           #        OUTPUT_W x OUTPUT_H), no extra border. Nice for presentations.
SCALE_MODE = "fit"         # "fit"       : every ROI is scaled to fill the frame -> same pixels/cm in all videos
                           #               if the arena has the same real size (recommended)
                           # "same_scale": all videos shrunk by the same factor (ROIs that look smaller in the
                           #               camera stay smaller, with more border around them)

# --- what goes outside the ROI ---
BACKGROUND = "black"       # colour outside the ROI and in the border: "black" (best for DLC), "white",
                           # any colour name ("gray", "navy"...) or a hex code like "#3060A0".
                           # "transparent": no background at all (for presentations / video editing).
                           #   MP4 can't store transparency, so these videos are saved as:
TRANSPARENT_FORMAT = "mov" # "mov" : ProRes 4444 with transparency – Keynote, Final Cut, Premiere, After Effects
                           #         (large files, best quality)
                           # "webm": VP9 with transparency – web browsers / web pages (small files)

# --- quality and speed ---
CRF        = 20            # MP4 quality: 18 = almost lossless, 23 = smaller files (18-22 is good for DLC)
PRESET     = "veryfast"    # MP4 encoding speed ("medium" = slightly smaller files but slower)
HWACCEL    = True          # use the Mac's hardware video decoder (falls back to CPU automatically)
N_PARALLEL = 2             # how many videos are processed at the same time (2-3 is usually fastest)
OVERWRITE  = False         # True = re-make videos that already exist in the output folder

# ---------------- folder picker (only if INPUT_DIR is empty) ----------------
# The dialog runs in a separate process, so it can't freeze the Jupyter kernel.
import sys, subprocess
def pick_folder(title="Select the folder with the videos"):
    if sys.platform == "darwin":   # native macOS Finder dialog
        r = subprocess.run(["osascript", "-e", f'POSIX path of (choose folder with prompt "{title}")'],
                           capture_output=True, text=True)
        return r.stdout.strip()     # "" if cancelled
    code = ("import tkinter as tk; from tkinter import filedialog; r = tk.Tk(); r.withdraw(); "
            "r.attributes('-topmost', True); "
            f"print(filedialog.askdirectory(title={title!r})); r.destroy()")
    r = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True)
    return r.stdout.strip()

if not INPUT_DIR:
    try:
        INPUT_DIR = pick_folder()
    except Exception as e:
        print("Folder dialog not available here:", e, "-> set INPUT_DIR by hand.")
    if not INPUT_DIR:
        print("No folder selected -> set INPUT_DIR above (or run this cell again).")
print("INPUT_DIR =", INPUT_DIR)

## Core functions (just run)

In [ ]:
# ---------- core helpers (no user settings here) ----------
import os, sys, re, io, json, shutil, subprocess, math, random
from pathlib import Path
import numpy as np
from PIL import Image

def find_ffmpeg():
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg
        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError:
        raise RuntimeError("ffmpeg not found. Install it (e.g. `conda install -c conda-forge ffmpeg`) "
                           "or `pip install imageio-ffmpeg`.")

FFMPEG = find_ffmpeg()

def list_videos(folder, exts):
    folder = Path(folder)
    return sorted([p for p in folder.iterdir()
                   if p.is_file() and p.suffix.lower() in exts and not p.name.startswith(".")])

def video_info(path):
    """Duration (s) and fps parsed from `ffmpeg -i` (no ffprobe needed)."""
    r = subprocess.run([FFMPEG, "-hide_banner", "-i", str(path)],
                       capture_output=True, text=True, errors="replace")
    txt = r.stderr
    dur = None
    m = re.search(r"Duration:\s*(\d+):(\d+):(\d+(?:\.\d+)?)", txt)
    if m:
        h, mi, s = m.groups(); dur = int(h) * 3600 + int(mi) * 60 + float(s)
    fps = None
    m = re.search(r"Video:.*?(\d+(?:\.\d+)?)\s*fps", txt)
    if m:
        fps = float(m.group(1))
    codec = pix = None
    m = re.search(r"Video:\s*(\w+)[^,]*,\s*(\w+)", txt)
    if m:
        codec, pix = m.groups()
    size = None
    m = re.search(r"Video:.*?,\s(\d{2,5})x(\d{2,5})", txt)
    if m:
        size = (int(m.group(1)), int(m.group(2)))
        if re.search(r"rotat\w*\D{0,20}-?(90|270)", txt):      # phone-style rotated video
            size = size[::-1]
    return {"duration": dur, "fps": fps, "codec": codec, "pix_fmt": pix, "size": size}

def grab_frame(path, t=0.0, alpha=False):
    """Return one frame (RGB uint8 array, full resolution, rotation applied) at time t (s)."""
    dec = ["-c:v", "libvpx-vp9"] if alpha and str(path).lower().endswith(".webm") else []  # reads VP9 alpha
    cmd = [FFMPEG, "-hide_banner", "-loglevel", "error", "-ss", f"{max(t, 0):.3f}", *dec, "-i", str(path),
           "-frames:v", "1", "-f", "image2pipe", "-vcodec", "png", "-"]
    r = subprocess.run(cmd, capture_output=True)
    if not r.stdout:  # t beyond the end -> fall back to the first frame
        cmd[cmd.index("-ss") + 1] = "0"
        r = subprocess.run(cmd, capture_output=True)
    if not r.stdout:
        raise RuntimeError(f"Could not read a frame from {path}:\n{r.stderr.decode(errors='replace')}")
    return np.array(Image.open(io.BytesIO(r.stdout)).convert("RGBA" if alpha else "RGB"))

def grab_preview(path, t, width=1280, hwaccel=True):
    """Fast small frame near time t: the keyframe at/before t, decoded alone and scaled to `width` px.
    (Uses the Mac hardware decoder when available.) Returns RGB array or None."""
    def run(hw):
        cmd = [FFMPEG, "-hide_banner", "-loglevel", "error"]
        if hw:
            cmd += ["-hwaccel", "videotoolbox" if sys.platform == "darwin" else "auto"]
        cmd += ["-noaccurate_seek", "-ss", f"{max(t, 0):.3f}", "-i", str(path), "-an", "-frames:v", "1",
                "-vf", f"scale={width}:-2:flags=fast_bilinear", "-f", "image2pipe", "-vcodec", "bmp", "-"]
        r = subprocess.run(cmd, capture_output=True)
        return np.array(Image.open(io.BytesIO(r.stdout)).convert("RGB")) if r.stdout else None
    f = run(hwaccel)
    return f if f is not None or not hwaccel else run(False)

def bisect_order(n):
    """0, n-1, middle, quarters, ... : broad coverage of the video first."""
    order, seen, queue = [], set(), [(0, n - 1)]
    for k in (0, n - 1):
        if k not in seen: order.append(k); seen.add(k)
    while queue:
        a, b = queue.pop(0)
        m = (a + b) // 2
        if m not in seen: order.append(m); seen.add(m)
        if m - a > 1: queue.append((a, m))
        if b - m > 1: queue.append((m, b))
    return order + [k for k in range(n) if k not in seen]

def fit_circle(pts):
    """Least-squares circle through >=3 points (x, y). Returns cx, cy, r."""
    pts = np.asarray(pts, float)
    x, y = pts[:, 0], pts[:, 1]
    A = np.c_[x, y, np.ones_like(x)]
    b = x ** 2 + y ** 2
    c, *_ = np.linalg.lstsq(A, b, rcond=None)
    cx, cy = c[0] / 2, c[1] / 2
    r = math.sqrt(max(c[2] + cx ** 2 + cy ** 2, 0))
    return float(cx), float(cy), float(r)

SHAPES = ("circle", "rectangle", "square", "polygon")

def load_rois(path, legacy_paths=()):
    """{video name: roi}. If `path` doesn't exist yet, the first existing legacy file is read
    (old circles.json files become circle ROIs)."""
    path = Path(path)
    src = path if path.exists() else next((Path(q) for q in legacy_paths if Path(q).exists()), None)
    rois = json.loads(src.read_text()) if src else {}
    for r in rois.values():
        r.setdefault("shape", "circle")
    return rois

def save_rois(rois, path):
    Path(path).write_text(json.dumps(rois, indent=2))

def even(n):
    n = int(round(n))
    return n if n % 2 == 0 else n + 1

def roi_from_points(shape, pts, W, H):
    """Build an ROI dict from clicked points (original-video pixels). None if not enough points."""
    pts = [(float(x), float(y)) for x, y in pts]
    if shape == "circle" and len(pts) >= 3:
        cx, cy, r = fit_circle(pts)
        roi = dict(cx=cx, cy=cy, r=r)
    elif shape in ("rectangle", "square") and len(pts) >= 2:
        (xa, ya), (xb, yb) = pts[-2:]
        if shape == "square":   # side = larger of the two extents, anchored at the first corner
            side = max(abs(xb - xa), abs(yb - ya))
            xb = xa + side * (1 if xb >= xa else -1)
            yb = ya + side * (1 if yb >= ya else -1)
        roi = dict(x0=min(xa, xb), y0=min(ya, yb), x1=max(xa, xb), y1=max(ya, yb))
    elif shape == "polygon" and len(pts) >= 3:
        roi = dict(pts=[[round(x, 2), round(y, 2)] for x, y in pts])
    else:
        return None
    roi = {k: (round(v, 2) if isinstance(v, float) else v) for k, v in roi.items()}
    return dict(shape=shape, **roi, W=int(W), H=int(H))

def roi_bbox(roi):
    """Float bounding box (x0, y0, x1, y1) of the ROI in original pixels."""
    s = roi["shape"]
    if s == "circle":
        return roi["cx"] - roi["r"], roi["cy"] - roi["r"], roi["cx"] + roi["r"], roi["cy"] + roi["r"]
    if s in ("rectangle", "square"):
        return roi["x0"], roi["y0"], roi["x1"], roi["y1"]
    a = np.array(roi["pts"], float)
    return a[:, 0].min(), a[:, 1].min(), a[:, 0].max(), a[:, 1].max()

def crop_box(roi):
    """Crop rectangle in original pixels (even size/offsets): (x0, y0, w, h). x0/y0 may be < 0."""
    bx0, by0, bx1, by1 = roi_bbox(roi)
    if roi["shape"] == "circle":     # keep the circle exactly centred in a square
        D = even(2 * roi["r"])
        x0, y0, w, h = int(round(roi["cx"] - D / 2)), int(round(roi["cy"] - D / 2)), D, D
    else:
        x0, y0 = int(math.floor(bx0)), int(math.floor(by0))
        w, h = even(bx1 - x0), even(by1 - y0)
    return x0 - x0 % 2, y0 - y0 % 2, w, h

def plan_sizes(rois, names, out_w, out_h, mode):
    """Size (w, h) in px of each video's ROI inside the output frame."""
    boxes = {n: crop_box(rois[n])[2:] for n in names}
    fit = {n: min(out_w / w, out_h / h) for n, (w, h) in boxes.items()}
    if mode == "fit":
        k = fit
    elif mode == "same_scale":
        kk = min(fit.values()); k = {n: kk for n in names}
    else:
        raise ValueError("SCALE_MODE must be 'fit' or 'same_scale'")
    return {n: (min(even(w * k[n] - 1), out_w), min(even(h * k[n] - 1), out_h)) for n, (w, h) in boxes.items()}

def parse_color(c):
    """'black', 'white', '#RRGGBB', any CSS colour name -> (r, g, b)."""
    from PIL import ImageColor
    return ImageColor.getrgb(c)[:3]

def make_mask(roi, size, path, background="black"):
    """Mask image at the output ROI size. Returns None for rectangles/squares (nothing to mask).
    - colour background: RGBA, colour outside the ROI, transparent inside (overlaid on the video)
    - "transparent":     grayscale, white inside, black outside (used as the video's alpha channel)"""
    from PIL import ImageDraw, ImageChops
    sw, sh = size
    x0, y0, w, h = crop_box(roi)
    W, H = roi["W"], roi["H"]
    off_frame = x0 < 0 or y0 < 0 or x0 + w > W or y0 + h > H     # ROI partly outside the camera image
    transparent = background == "transparent"
    if roi["shape"] in ("rectangle", "square") and not (transparent and off_frame):
        return None                                   # plain crop, the border colour is added by ffmpeg
    ss = 4                                            # supersampling for smooth edges
    kx, ky = sw / w * ss, sh / h * ss
    m = Image.new("L", (sw * ss, sh * ss), 0)
    d = ImageDraw.Draw(m)
    if roi["shape"] == "circle":
        cx, cy, r = (roi["cx"] - x0) * kx, (roi["cy"] - y0) * ky, roi["r"]
        d.ellipse([cx - r * kx, cy - r * ky, cx + r * kx, cy + r * ky], fill=255)
    elif roi["shape"] == "polygon":
        d.polygon([((x - x0) * kx, (y - y0) * ky) for x, y in roi["pts"]], fill=255)
    else:
        d.rectangle([0, 0, sw * ss, sh * ss], fill=255)
    if transparent and off_frame:                     # outside the camera image -> transparent too
        valid = Image.new("L", m.size, 0)
        ImageDraw.Draw(valid).rectangle([-x0 * kx, -y0 * ky, (W - x0) * kx - 1, (H - y0) * ky - 1], fill=255)
        m = ImageChops.darker(m, valid)
    inside = m.resize((sw, sh), Image.LANCZOS)          # 255 inside the ROI
    if background == "transparent":
        inside.save(path)
    else:
        rgba = Image.new("RGBA", (sw, sh), parse_color(background) + (0,))
        rgba.putalpha(inside.point(lambda v: 255 - v))   # opaque outside
        rgba.save(path)
    return path

def roi_patch(roi, **style):
    """matplotlib patch drawing the ROI."""
    import matplotlib.patches as mp
    s = roi["shape"]
    if s == "circle":
        return mp.Circle((roi["cx"], roi["cy"]), roi["r"], fill=False, **style)
    if s in ("rectangle", "square"):
        return mp.Rectangle((roi["x0"], roi["y0"]), roi["x1"] - roi["x0"], roi["y1"] - roi["y0"], fill=False, **style)
    return mp.Polygon(roi["pts"], closed=True, fill=False, **style)

def decode_modes(hwaccel):
    """Decoding strategies, fastest first. Each = (name, ffmpeg input args, filter inserted after fps)."""
    modes = []
    if hwaccel and sys.platform == "darwin":
        # frames stay in the Apple hardware decoder; only the frames kept by fps= are copied to RAM
        modes.append(("gpu", ["-hwaccel", "videotoolbox", "-hwaccel_output_format", "videotoolbox_vld"], "hwdownload"))
    if hwaccel:
        modes.append(("hw", ["-hwaccel", "auto"], None))
    modes.append(("cpu", [], None))
    return modes

_BEST_MODE = {}   # remembers which decoding strategy works on this machine

def build_filter(roi, size, out_size, fps, masked=True, background="black", after_fps=None, pix_fmt=None):
    """ffmpeg graph: fps -> [hw download] -> (pad if ROI leaves frame) -> crop -> scale -> [mask] -> [centre pad].
    out_size=None -> no padding (output = just the ROI box)."""
    W, H = roi["W"], roi["H"]
    x0, y0, w, h = crop_box(roi)
    sw, sh = size
    transparent = background == "transparent"
    rgb = (0, 0, 0) if transparent else parse_color(background)
    hexcol = "0x%02X%02X%02X" % rgb
    p = max(0, -x0, -y0, x0 + w - W, y0 + h - H)
    p += p % 2
    chain = []
    if fps:
        chain.append(f"fps={fps}")
    if after_fps == "hwdownload":
        chain += ["hwdownload", "format=p010le" if pix_fmt and "10" in pix_fmt else "format=nv12"]
    if p > 0:   # ROI partly outside the camera frame: that part gets the background colour
        chain.append(f"pad=iw+{2 * p}:ih+{2 * p}:{p}:{p}:{hexcol}")
        x0 += p; y0 += p
    chain += [f"crop={w}:{h}:{x0}:{y0}", f"scale={sw}:{sh}:flags=area", "setsar=1"]
    fmt = "yuva444p" if transparent else "yuv420p"
    chain.append(f"format={fmt}" if not transparent else "format=yuv444p")
    tail = []
    if out_size and tuple(out_size) != (sw, sh):
        ow, oh = out_size
        tail.append(f"pad={ow}:{oh}:{(ow - sw) // 2}:{(oh - sh) // 2}:{'black@0' if transparent else hexcol}")
    tail.append(f"format={fmt}")
    tail = ",".join(tail)
    if transparent:
        if masked:
            return f"[0:v]{','.join(chain)}[v];[1:v]format=gray[m];[v][m]alphamerge,{tail}[out]"
        return f"[0:v]{','.join(chain)},format=yuva444p,{tail}[out]"
    if masked:
        return f"[0:v]{','.join(chain)}[v];[v][1:v]overlay=0:0:format=auto,{tail}[out]"
    return f"[0:v]{','.join(chain)},{tail}[out]"

# encoder settings per output container
def encoder_args(ext, crf=20, preset="veryfast"):
    if ext == ".mov":    # ProRes 4444 with alpha (Keynote, Final Cut, Premiere, After Effects)
        return ["-c:v", "prores_ks", "-profile:v", "4444", "-pix_fmt", "yuva444p10le", "-vendor", "apl0"]
    if ext == ".webm":   # VP9 with alpha (web browsers)
        return ["-c:v", "libvpx-vp9", "-pix_fmt", "yuva420p", "-crf", str(crf + 10), "-b:v", "0",
                "-deadline", "good", "-cpu-used", "4", "-row-mt", "1", "-auto-alt-ref", "0"]
    return ["-c:v", "libx264", "-preset", preset, "-crf", str(crf), "-pix_fmt", "yuv420p",
            "-movflags", "+faststart"]

_RUNNING = set()   # live ffmpeg processes (killed if you interrupt the cell)

def stop_all():
    for p in list(_RUNNING):
        try: p.kill()
        except Exception: pass

def process_video(src, dst, roi, size, out_size, fps, background="black", crf=20, preset="veryfast",
                  hwaccel=True, mask_dir=None, log_dir=None, pbar=None):
    """Crop + mask + downsample one video. out_size=None -> no padding. Returns (ok, mode_used).
    The container (and codec) follows dst's extension: .mp4 (H.264), .mov (ProRes 4444 + alpha), .webm (VP9 + alpha)."""
    mask_dir = Path(mask_dir or Path(dst).parent)
    log_dir = Path(log_dir or Path(dst).parent)
    mask = make_mask(roi, size, mask_dir / f"_mask_{Path(src).stem}.png", background)
    info = video_info(src)
    dur = info["duration"] or 0
    tmp = Path(dst).with_name(Path(dst).stem + ".partial" + Path(dst).suffix)
    log = log_dir / (Path(src).stem + ".log")

    def run(name, in_args, after_fps):
        graph = build_filter(roi, size, out_size, fps, mask is not None, background, after_fps, info["pix_fmt"])
        cmd = [FFMPEG, "-y", "-hide_banner", "-nostats", "-progress", "pipe:1", *in_args,
               "-i", str(src), *(["-i", str(mask)] if mask else []), "-filter_complex", graph,
               "-map", "[out]", "-an", *encoder_args(Path(dst).suffix.lower(), crf, preset), str(tmp)]
        with open(log, "a") as lf:
            lf.write(f"\n===== mode: {name} =====\n" + " ".join(cmd) + "\n\n"); lf.flush()
            proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=lf, text=True)
            _RUNNING.add(proc)
            speed = ""
            try:
                for line in proc.stdout:
                    key, _, val = line.strip().partition("=")
                    if key == "speed":
                        speed = val.strip()
                    elif key == "out_time_us" and pbar is not None:
                        try:
                            t = int(val) / 1e6
                        except ValueError:
                            continue
                        if 0 <= t <= dur + 5:          # ffmpeg reports a placeholder before the 1st frame
                            pbar.n = round(min(t, dur), 1) if dur else round(t, 1)
                            pbar.set_postfix_str(f"{speed} real-time · {name}", refresh=False)
                            pbar.refresh()
                proc.wait()
            finally:
                if proc.poll() is None:
                    proc.kill()
                _RUNNING.discard(proc)
        return proc.returncode == 0

    log.write_text("")
    modes = decode_modes(hwaccel)
    best = _BEST_MODE.get("mode")
    if best:   # start with the strategy that already worked on a previous video
        modes = [m for m in modes if m[0] == best] + [m for m in modes if m[0] != best]
    for name, in_args, after_fps in modes:
        if pbar is not None:
            pbar.n = 0; pbar.refresh()
        if run(name, in_args, after_fps):
            _BEST_MODE["mode"] = name
            os.replace(tmp, dst)
            if pbar is not None:
                pbar.n = pbar.total or pbar.n
                pbar.set_postfix_str(f"done · {name}"); pbar.refresh()
            return True, name
    if tmp.exists():
        tmp.unlink()
    return False, None

# ---------- set up folders ----------
INPUT_DIR = Path(INPUT_DIR).expanduser()
assert INPUT_DIR.is_dir(), f"Folder not found: {INPUT_DIR}"
OUTPUT_DIR = Path(OUTPUT_DIR).expanduser() if OUTPUT_DIR else INPUT_DIR / "DLC_ready"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "_logs").mkdir(exist_ok=True)
ROIS_JSON = INPUT_DIR / "rois.json"   # next to the videos -> shared by every output folder
assert OUTPUT_W % 2 == 0 and OUTPUT_H % 2 == 0, "OUTPUT_W and OUTPUT_H must be even"

VIDEOS = list_videos(INPUT_DIR, VIDEO_EXTS)
LEGACY = [OUTPUT_DIR / "rois.json", INPUT_DIR / "DLC_ready" / "rois.json", INPUT_DIR / "DLC_ready" / "circles.json"]
rois = load_rois(ROIS_JSON, LEGACY)
if rois and not ROIS_JSON.exists():
    save_rois(rois, ROIS_JSON)       # older notebook versions kept them in the output folder
assert ROI_SHAPE in SHAPES, f"ROI_SHAPE must be one of {SHAPES}"
if BACKGROUND != "transparent":
    parse_color(BACKGROUND)   # raises an error now if the colour name is not valid
assert TRANSPARENT_FORMAT in ("mov", "webm"), 'TRANSPARENT_FORMAT must be "mov" or "webm"'
OUT_EXT = ("." + TRANSPARENT_FORMAT) if BACKGROUND == "transparent" else ".mp4"
print(f"ffmpeg: {FFMPEG}")
print(f"{len(VIDEOS)} videos in {INPUT_DIR}  ->  output: {OUTPUT_DIR}")
for v in VIDEOS:
    print(f"  {'✔ ' + rois[v.name]['shape'] if v.name in rois else '·'} {v.name}  ({v.stat().st_size/1e9:.2f} GB)")

## Step 1 · Draw the area of interest (ROI)

Each video opens one after the other. For each one:

1. **Pick the shape** in the *shape* menu (it starts at `ROI_SHAPE`).
2. **Click on the image** (left click):

   | shape | how to draw |
   |---|---|
   | **circle** | click **3 or more points on the edge**, spread around it (more points = more precise). The red circle is fitted through them. |
   | **rectangle** | click **2 opposite corners**. Clicking again moves the nearest corner. |
   | **square** | click **2 opposite corners** (the side = the longer of the two). Clicking again moves the nearest corner. |
   | **polygon** (custom) | click the **corners in order** around the area (3 or more). It closes by itself. |

3. Press **Save & next** (green). If you didn't click anything, the ROI from the previous video (blue dashed) is used — handy when the camera didn't move.

**Helpers**
- **Frame slider / ◀ ▶** — look at different moments of the video (start → end) to check the camera or arena didn't move. Small preview frames load in the background (counter next to the buttons); the next video loads while you work. Your clicks and zoom stay when you change frame.
- **🔍 Full resolution** — shows the current frame at full size, to place points precisely.
- **Zoom / pan** — toolbar of the image. Turn the zoom tool **off** again before clicking points.
- **Undo** removes the last point, **Clear** all of them, **Skip** leaves this video without an ROI (it won't be processed).
- Already-drawn videos are skipped. To redraw them, set `REDO = True` below and run the cell again.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor

REDO          = False   # True -> go through all videos again, not only the ones without an ROI
PREVIEW_N     = 40      # preview frames per video (spread evenly from start to end)
PREVIEW_WIDTH = 1280    # preview width in px

class ROISelector:
    def __init__(self, videos, rois, save_path, redo=False, shape="circle"):
        self.rois, self.save_path = rois, save_path
        self.queue = [v for v in videos if redo or v.name not in rois]
        self.i, self.pts, self.prev, self.im, self._mute = 0, [], None, None, False
        self.cache, self.meta, self.futs = {}, {}, {}
        self.pool = ThreadPoolExecutor(max_workers=2)
        if rois:  # last saved ROI = suggestion for the next video
            self.prev = list(rois.values())[-1]
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=(8, 8))
        plt.ion()
        self.fig.canvas.header_visible = False
        self.fig.canvas.mpl_connect("button_press_event", self.on_click)
        self.title, self.status, self.tlabel, self.loadlabel = W.HTML(), W.HTML(), W.HTML(), W.HTML()
        self.slider = W.IntSlider(min=0, max=1, value=0, description="frame", continuous_update=True,
                                  readout=False, layout=W.Layout(width="55%"))
        self.slider.observe(self._on_slider, names="value")
        def btn(desc, cb, style="", width=None):
            b = W.Button(description=desc, button_style=style,
                         layout=W.Layout(width=width) if width else W.Layout())
            b.on_click(cb); return b
        self.nav = W.HBox([btn("◀", lambda _: self.step(-1), width="40px"), self.slider,
                           btn("▶", lambda _: self.step(1), width="40px"), self.tlabel,
                           btn("🔍 Full resolution", self.full_res)])
        self.shape = W.Dropdown(options=list(SHAPES), value=shape, description="shape",
                                layout=W.Layout(width="200px"))
        self.shape.observe(lambda ch: self.clear(), names="value")
        self.buttons = W.HBox([self.shape, btn("Undo", self.undo), btn("Clear", self.clear),
                               btn("Skip", self.skip, "warning"), btn("Save & next", self.save_next, "success"),
                               self.loadlabel])
        display(W.VBox([self.title, self.nav, self.buttons, self.status, self.fig.canvas]))
        if not self.queue:
            self.title.value = "<b>All videos already have an ROI.</b> Set REDO = True to redraw."
            self.nav.layout.display = self.buttons.layout.display = "none"
            self.fig.canvas.layout.display = "none"; plt.close(self.fig)
        else:
            self.load_video()

    # --- background preview loading ---
    def prepare(self, vi):
        if vi in self.meta or vi >= len(self.queue):
            return
        v = self.queue[vi]
        info = video_info(v)
        dur = info["duration"] or 0
        info["times"] = list(np.linspace(0, max(dur - 1.0, 0), PREVIEW_N)) if dur > 1 else [0.0]
        self.meta[vi] = info
        self.futs[vi] = [self.pool.submit(self._load, vi, k) for k in bisect_order(len(info["times"]))]

    def _load(self, vi, k):
        if (vi, k) in self.cache or vi not in (self.i, self.i + 1):
            return
        f = grab_preview(self.queue[vi], self.meta[vi]["times"][k], PREVIEW_WIDTH, HWACCEL)
        self.cache[(vi, k)] = f
        if vi == self.i:
            n = len(self.meta[vi]["times"]); done = sum(1 for (a, _) in self.cache if a == vi)
            self.loadlabel.value = (f"&nbsp;&nbsp;<small>previews loaded {done}/{n}</small>" if done < n
                                    else "&nbsp;&nbsp;<small>all previews loaded ✔ (pre-loading next video)</small>")

    def get_frame(self, k):
        f = self.cache.get((self.i, k))
        if f is None:   # not loaded yet -> load it now
            f = grab_preview(self.queue[self.i], self.meta[self.i]["times"][k], PREVIEW_WIDTH, HWACCEL)
            self.cache[(self.i, k)] = f
        return f

    # --- video / frame display ---
    def load_video(self):
        for vi in list(self.futs):                       # forget older videos
            if vi < self.i:
                for fu in self.futs.pop(vi): fu.cancel()
        for key in [key for key in self.cache if key[0] < self.i]:
            del self.cache[key]
        self.prepare(self.i)
        self.prepare(self.i + 1)                          # pre-load the next video in the background
        v, info = self.queue[self.i], self.meta[self.i]
        self.pts, self.im = [], None
        n = len(info["times"])
        self._mute = True
        self.slider.max = max(n - 1, 0); self.slider.value = 0
        self._mute = False
        self.title.value = (f"<b>{self.i+1}/{len(self.queue)} · {v.name}</b> — {info['codec']}, "
                            f"{info['fps']} fps, {info['duration'] or 0:.0f} s")
        self.show_frame(0)

    def _draw(self, frame, label):
        if self.im is None:    # first frame of this video: set up the axes in ORIGINAL pixel coordinates
            size = self.meta[self.i]["size"]
            self.W, self.H = size if size else grab_frame(self.queue[self.i], 0).shape[1::-1]
            self.ax.clear()
            self.im = self.ax.imshow(frame, extent=(0, self.W, self.H, 0))
            self.ax.set_xlim(0, self.W); self.ax.set_ylim(self.H, 0)
            self.ax.set_axis_off(); self.fig.tight_layout()
        else:                  # keep zoom and points
            self.im.set_data(frame)
        self.tlabel.value = label
        self.redraw()

    def show_frame(self, k):
        f = self.get_frame(k)
        if f is None:
            self.status.value = "<span style='color:red'>could not read this frame</span>"; return
        t, dur = self.meta[self.i]["times"][k], self.meta[self.i]["duration"] or 0
        self._draw(f, f"&nbsp;{k+1}/{len(self.meta[self.i]['times'])} · ≈{t:.0f} s / {dur:.0f} s")

    def full_res(self, _=None):
        self.status.value = "<i>loading full-resolution frame…</i>"
        t = self.meta[self.i]["times"][self.slider.value]
        self._draw(grab_frame(self.queue[self.i], t), f"&nbsp;{t:.1f} s · <b>full resolution</b>")

    def _on_slider(self, change):
        if not self._mute:
            self.show_frame(change["new"])

    def step(self, d):
        self.slider.value = int(np.clip(self.slider.value + d, 0, self.slider.max))

    # --- ROI ---
    HELP = {"circle": "click ≥3 points on the edge", "rectangle": "click 2 opposite corners",
            "square": "click 2 opposite corners", "polygon": "click the corners in order (≥3)"}

    def current(self):
        return roi_from_points(self.shape.value, self.pts, self.W, self.H)

    def redraw(self):
        for a in list(self.ax.patches) + list(self.ax.lines):
            a.remove()
        roi = self.current()
        if self.prev and roi is None:
            self.ax.add_patch(roi_patch(self.prev, color="deepskyblue", lw=1.5, ls="--"))
        if self.pts:
            xs, ys = zip(*self.pts)
            self.ax.plot(xs, ys, "o", color="yellow", mec="k", ms=6)
        if roi:
            self.ax.add_patch(roi_patch(roi, color="red", lw=2))
            x0, y0, x1, y1 = roi_bbox(roi)
            self.ax.plot([(x0 + x1) / 2], [(y0 + y1) / 2], "+", color="red", ms=12)
            info = f"{roi['shape']} · box {x1-x0:.0f}×{y1-y0:.0f} px"
            if roi["shape"] == "circle":
                res = np.abs(np.hypot(*(np.array(self.pts) - [roi["cx"], roi["cy"]]).T) - roi["r"]).max()
                info += f", radius {roi['r']:.0f} px, max point deviation {res:.1f} px"
            self.status.value = f"{len(self.pts)} points · {info}"
        else:
            msg = f"{self.shape.value}: {self.HELP[self.shape.value]} ({len(self.pts)} so far)."
            if self.prev:
                msg += " Blue dashed = previous ROI (used if you save now)."
            self.status.value = msg
        self.fig.canvas.draw_idle()

    # --- events ---
    def on_click(self, e):
        if e.inaxes != self.ax or e.button != 1 or self.ax.get_navigate_mode() is not None:
            return
        p = (e.xdata, e.ydata)
        if self.shape.value in ("rectangle", "square") and len(self.pts) >= 2:
            # move the nearest corner
            k = int(np.argmin([np.hypot(p[0] - x, p[1] - y) for x, y in self.pts[-2:]]))
            self.pts = self.pts[-2:]; self.pts[k] = p
        else:
            self.pts.append(p)
        self.redraw()
    def undo(self, _=None):
        if self.pts: self.pts.pop(); self.redraw()
    def clear(self, _=None):
        self.pts = []; self.redraw()
    def skip(self, _=None):
        self.next()
    def save_next(self, _=None):
        roi = self.current()
        if roi is None and self.prev is not None:
            roi = dict(self.prev, W=self.W, H=self.H)
        if roi is None:
            self.status.value = f"<span style='color:red'>{self.shape.value}: {self.HELP[self.shape.value]} first.</span>"; return
        self.rois[self.queue[self.i].name] = roi
        save_rois(self.rois, self.save_path)
        self.prev = roi
        self.next()
    def next(self):
        self.i += 1
        if self.i >= len(self.queue):
            self.pool.shutdown(wait=False, cancel_futures=True); self.cache.clear()
            self.fig.canvas.layout.display = "none"; plt.close(self.fig)
            self.title.value = f"<b>Done ✔</b> {len(self.rois)} ROIs saved in {self.save_path}"
            self.status.value = self.loadlabel.value = ""
            self.nav.layout.display = self.buttons.layout.display = "none"
        else:
            self.load_video()

selector = ROISelector(VIDEOS, rois, ROIS_JSON, redo=REDO, shape=ROI_SHAPE)

### Check the ROIs (optional)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
rois = load_rois(ROIS_JSON)
todo = [v for v in VIDEOS if v.name in rois]
n = len(todo); cols = min(4, max(n, 1)); rows = int(np.ceil(n / cols)) if n else 1
fig, axs = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for ax in axs.flat: ax.set_axis_off()
for ax, v in zip(axs.flat, todo):
    roi = rois[v.name]
    f = grab_preview(v, 1.0, 1280, HWACCEL)
    s = max(1, f.shape[1] // 800)
    ax.imshow(f, extent=(0, roi["W"], roi["H"], 0))
    ax.add_patch(roi_patch(roi, color="red", lw=2))
    ax.set_xlim(0, roi["W"]); ax.set_ylim(roi["H"], 0)
    ax.set_title(f"{v.name}\n{roi['shape']}", fontsize=9)
plt.tight_layout(); plt.show()
missing = [v.name for v in VIDEOS if v.name not in rois]
print("Videos without an ROI (will be skipped):", missing or "none")

## Step 2 · Crop, downsample and save all videos
Just run the cell. For each video, ffmpeg does everything in one pass: keep `TARGET_FPS` frames per second → crop the rectangle around the ROI → scale → fill outside the ROI with `BACKGROUND` → add a border up to `OUTPUT_W × OUTPUT_H` (if `UNIFORM_SIZE`).

- One progress bar per video: % done, time left, speed (× real time) and decoding method (`gpu` = fastest; it falls back to `hw` / `cpu` by itself if needed).
- `N_PARALLEL` videos run at the same time. Pressing ■ (stop) cancels all of them.
- Videos already in the output folder are skipped (`OVERWRITE = True` to redo them). If something fails, the reason is in `_logs/<video>.log`.

In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
plt.close('all')
rois = load_rois(ROIS_JSON)
todo = [v for v in VIDEOS if v.name in rois]
sizes = plan_sizes(rois, [v.name for v in todo], OUTPUT_W, OUTPUT_H, SCALE_MODE)

jobs = []
for v in todo:
    dst = OUTPUT_DIR / (v.stem + OUT_EXT)
    if dst.exists() and not OVERWRITE:
        print(f"{v.name}: already done, skipped"); continue
    roi, size = rois[v.name], sizes[v.name]
    if size[0] > crop_box(roi)[2]:
        print(f"note: {v.name} ROI is {crop_box(roi)[2]}px wide, smaller than {size[0]}px -> it will be upscaled")
    jobs.append((v, dst, roi, size))

# one progress bar per video (seconds of video processed)
bars = {v.name: tqdm(total=round(video_info(v)["duration"] or 0, 1), desc=v.name, unit="s",
                     bar_format="{desc}: {percentage:3.0f}%|{bar}| {n:.0f}/{total:.0f} s [{elapsed}<{remaining}{postfix}]")
        for v, *_ in jobs}

def job(args):
    v, dst, roi, size = args
    ok, mode = process_video(v, dst, roi, size, (OUTPUT_W, OUTPUT_H) if UNIFORM_SIZE else None, TARGET_FPS,
                             background=BACKGROUND, crf=CRF, preset=PRESET,
                             hwaccel=HWACCEL, mask_dir=OUTPUT_DIR / "_logs",
                             log_dir=OUTPUT_DIR / "_logs", pbar=bars[v.name])
    return v.name, ok

t0, failed = time.time(), []
try:
    with ThreadPoolExecutor(max_workers=max(1, N_PARALLEL)) as ex:
        for name, ok in ex.map(job, jobs):
            if not ok:
                failed.append(name)
except KeyboardInterrupt:
    stop_all(); print("Interrupted – all ffmpeg jobs stopped.")
for b in bars.values():
    b.close()

# record of what was done (to convert DLC coordinates back to the original videos)
params = {"output_size": [OUTPUT_W, OUTPUT_H] if UNIFORM_SIZE else "roi size", "fps": TARGET_FPS,
          "scale_mode": SCALE_MODE, "background": BACKGROUND, "videos": {}}
for v in todo:
    roi, (sw, sh) = rois[v.name], sizes[v.name]
    x0, y0, w, h = crop_box(roi)
    params["videos"][v.name] = dict(roi=roi, crop_origin_in_px=[x0, y0], crop_size_in_px=[w, h],
                                    roi_size_out_px=[sw, sh], scale_out_per_in=[sw / w, sh / h],
                                    roi_offset_out_px=[(OUTPUT_W - sw) // 2, (OUTPUT_H - sh) // 2] if UNIFORM_SIZE else [0, 0])
(OUTPUT_DIR / "processing_params.json").write_text(json.dumps(params, indent=2))
print(f"\nFinished in {(time.time()-t0)/60:.1f} min." +
      (f" FAILED: {failed} (see _logs)" if failed else (" All videos processed ✔" if jobs else "")))

## Step 3 · Check the new videos
Shows the middle frame of every new video with its size, fps, duration and file size. Transparent areas are shown as a grey checkerboard.

In [ ]:
%matplotlib inline
plt.close('all')
outs = sorted(p for p in OUTPUT_DIR.iterdir()
              if p.suffix.lower() in (".mp4", ".mov", ".webm") and ".partial" not in p.name)
def checker(h, w, n=16):
    yy, xx = np.mgrid[0:h, 0:w]
    return np.where(((yy // n + xx // n) % 2)[..., None] == 1, 200, 255).astype(float)
cols = min(4, max(len(outs), 1)); rows = int(np.ceil(len(outs) / cols)) if outs else 1
fig, axs = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for ax in axs.flat: ax.set_axis_off()
sizes_found = {}
for ax, p in zip(axs.flat, outs):
    info = video_info(p)
    f = grab_frame(p, (info["duration"] or 0) / 2, alpha=True).astype(float)
    a = f[..., 3:4] / 255
    ax.imshow((f[..., :3] * a + checker(*f.shape[:2]) * (1 - a)).astype(np.uint8))
    sizes_found[p.name] = (f.shape[1], f.shape[0])
    ax.set_title(f"{p.name}\n{f.shape[1]}×{f.shape[0]}, {info['fps']} fps, "
                 f"{info['duration']:.0f}s, {p.stat().st_size/1e6:.0f} MB", fontsize=9)
plt.tight_layout(); plt.show()
if UNIFORM_SIZE:
    bad = [n for n, sz in sizes_found.items() if sz != (OUTPUT_W, OUTPUT_H)]
    print("All videos have the same size ✔" if not bad else f"Different size: {bad}")

### Notes for DeepLabCut
- For DLC use the defaults: `BACKGROUND = "black"`, `UNIFORM_SIZE = True` (MP4 output). Add the videos from `DLC_ready/` to your project (`deeplabcut.create_new_project(..., videos=[...])` or `deeplabcut.add_new_videos`).
- With `SCALE_MODE = "fit"` the ROI fills the frame the same way in every video, so if the arena has the same real size everywhere, **one pixels → cm factor works for all videos** (ROI size in pixels = `roi_size_out_px` in `processing_params.json`).
- From DLC coordinates back to the original video: `x_orig = (x_out - roi_offset_x) / scale_x + crop_origin_x`, same for y (values per video in `processing_params.json`).
- Time: frame *i* of a new video is at `i / TARGET_FPS` seconds in the original video.
- Want a version for a presentation too? Set another `OUTPUT_DIR` (e.g. `"/Users/you/Desktop/for_slides"`), change `BACKGROUND` (e.g. `"transparent"` or `"white"`) and `UNIFORM_SIZE = False`, then re-run Settings, Core functions and Step 2. The ROIs are stored in `rois.json` next to the input videos, so you don't redraw anything.